# Tutorial: Obtaining and Modifying Molecular Conformers

This notebook demonstrates how to generate, visualize, and modify molecular conformers using RDKit and custom utilities to go back an forth between RDKit and AtomArrays (as well as save out to mmCIF files). Let's break it down step by step:

1. Importing Libraries and Setting Up Visualization
The notebook starts by importing necessary functions and setting up RDKit's IPythonConsole for better visualization in the Jupyter environment. This includes options for kekulization, atom indices, 3D rendering, and stereochemistry annotation.

In [ ]:
from datahub.transforms.rdkit_utils import *
from cifutils.utils.io_utils import to_cif_file
from cifutils.utils.visualize import view

try:
    # Settings for debugging & interactive tests
    from rdkit.Chem.Draw import IPythonConsole

    IPythonConsole.kekulizeStructures = False
    IPythonConsole.drawOptions.addAtomIndices = True
    IPythonConsole.ipython_3d = True
    IPythonConsole.ipython_useSVG = True
    IPythonConsole.drawOptions.addStereoAnnotation = True
    IPythonConsole.molSize = 600, 300
except ImportError:
    pass

2. Creating and Preparing the Molecule

In [ ]:
smiles = "c1(ccc2c(c1)oc1-c(c2c2cc(ccc2C(=O)[O-])C(=O)NC[C@H](Oc2cc3c(cc2)c(cc(=O)o3)C)O)ccc(=[N+]2CCC2)c1)N1CCC1"
mol = smiles_to_rdkit(smiles)
mol

In [ ]:
# Explicitly add all hydrogens
mol = add_hydrogens(mol)
mol

Let's have a look at the docstring for generate_conformers, to pick the best options for us:

In [ ]:
generate_conformers?

In [ ]:
mol_conf = generate_conformers(mol, optimize=True, infer_hydrogens=False, n_conformers=10)
mol_conf

Per default this visualizes the first conformer. We generated 10 conformers, so let's visualize the 10th one.

In [ ]:
IPythonConsole.drawMol3D(mol_conf, confId=9)

Excellent, we now want to modify this conformer slightly to be in the desired protonation and charge (formal charge) state. 
RDKit per default always tries to neutralize molecules, so we will need to manually do some of these modificaionts. 
Finally, we will save it out as a cif file.


In [ ]:
# Save out as a cif file
conf_id = 0  # <-- which conformer to save out
arr = atom_array_from_rdkit(mol_conf, elements_as_int=False, conformer_id=conf_id, remove_hydrogens=False)

# Add some annotations for the cif file
arr.chain_id = ["A"] * arr.array_length()
arr.res_name = ["UNL"] * arr.array_length()
arr.atom_name = np.char.add(arr.element.astype(str), np.arange(arr.array_length()).astype(str))

# Remove two of the hydrogens and set the charge on O37 to negative
arr = arr[~np.isin(arr.atom_name, ["H67", "H59"])]
arr.charge[arr.atom_name == "O37"] = -1

# Save out as a cif file
to_cif_file(arr, f"./jf1_conf{conf_id}.cif", id=f"jf1_conf{conf_id}")

Let's visualize the cif file to make sure it looks correct and that the hydrogen was removed properly.

In [ ]:
from cifutils.utils.visualize import view

view(arr)
# Hover over the atoms to see the chain, residue and atom names (and index).